# LoRA From the Ground Up — The Math, the Matrices, and the Merge

Companion notebook for the blog post at [cmenguy.github.io](https://cmenguy.github.io/llm/fine-tuning/2025/12/28/lora-deep-dive/).

This notebook walks through LoRA (Low-Rank Adaptation) from scratch — implementing the core algorithm in PyTorch, training a real model, inspecting the matrices, and demonstrating the merge process.

## Setup

Install all dependencies. This notebook runs on CPU for the pure math sections, and optionally on GPU for the model training section.

In [ ]:
!pip install -q torch numpy transformers peft trl datasets

## The Problem: Parameter Count in Full Fine-Tuning

Full fine-tuning updates every parameter. For Adam optimizer, that means 3x model size in memory (weights + gradients + optimizer states).

In [ ]:
# Memory requirements for full fine-tuning
model_params = 8e9  # 8B parameters (Llama 3.1 8B)
bytes_per_param = 4  # float32
optimizer_multiplier = 3  # weight + gradient + adam states

memory_gb = model_params * bytes_per_param * optimizer_multiplier / 1e9
print(f"Full fine-tuning memory (optimizer only): {memory_gb:.0f} GB")
print(f"That's before activations, batch data, etc.")

## LoRA Parameter Savings

LoRA decomposes the weight update $\Delta W = BA$ where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$ with rank $r \ll \min(d, k)$.

Let's see the parameter savings for a single layer.

In [ ]:
d, k, r = 4096, 4096, 8
full_params = d * k
lora_params = (d * r) + (r * k)
print(f"Full fine-tuning: {full_params:,} parameters")
print(f"LoRA (r={r}):     {lora_params:,} parameters")
print(f"Reduction:        {full_params / lora_params:.0f}x")

## LoRA Implementation From Scratch

Let's implement LoRA as a PyTorch module to understand exactly what happens during the forward pass.

The forward pass computes: $h = W_0 x + \frac{\alpha}{r} \cdot BAx$

In [ ]:
import torch
import torch.nn as nn


class LoRALinear(nn.Module):
    def __init__(self, original_layer, r=8, lora_alpha=16):
        super().__init__()
        self.original = original_layer
        self.original.weight.requires_grad = False  # freeze

        d, k = original_layer.weight.shape
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r

        # A is initialized with Kaiming uniform (like the paper)
        self.lora_A = nn.Parameter(torch.empty(r, k))
        nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)

        # B is initialized to zero so ΔW = BA = 0 at start
        self.lora_B = nn.Parameter(torch.zeros(d, r))

    def forward(self, x):
        # Original frozen path
        base_output = self.original(x)
        # LoRA path: x → A → B → scale
        lora_output = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_output + lora_output

### Verify the initialization

At initialization, $B = 0$ so $\Delta W = BA = 0$. The model output should be identical to the original.

In [ ]:
torch.manual_seed(42)

# Create original layer and LoRA-wrapped version
original = nn.Linear(128, 256, bias=False)
lora_layer = LoRALinear(original, r=8, lora_alpha=16)

# Test with random input
x = torch.randn(4, 128)
original_out = original(x)
lora_out = lora_layer(x)

print(f"Original output norm: {original_out.norm():.4f}")
print(f"LoRA output norm:     {lora_out.norm():.4f}")
print(f"Outputs identical:    {torch.allclose(original_out, lora_out)}")
print(f"\nTrainable params:    {sum(p.numel() for p in lora_layer.parameters() if p.requires_grad):,}")
print(f"Frozen params:       {sum(p.numel() for p in lora_layer.parameters() if not p.requires_grad):,}")

## Understanding Rank

The rank $r$ controls how many independent directions the weight update can modify. Let's see how energy concentrates in the singular values of a random vs. structured update.

In [ ]:
import torch

torch.manual_seed(42)
d, k = 4096, 4096

# Simulate a fine-tuning weight update
delta_W = torch.randn(d, k) * 0.01  # small perturbation
U, S, Vt = torch.linalg.svd(delta_W, full_matrices=False)

# How much "energy" is captured by the top-r singular values?
total_energy = (S ** 2).sum()
for r in [1, 2, 4, 8, 16, 32, 64]:
    captured = (S[:r] ** 2).sum() / total_energy * 100
    print(f"r={r:3d}: {captured:.2f}% of energy captured")

Now let's see what happens with a *structured* (low-rank) update — closer to what real fine-tuning produces:

In [ ]:
# Simulate a structured update (inherently low-rank)
true_rank = 4
B_true = torch.randn(d, true_rank) * 0.01
A_true = torch.randn(true_rank, k) * 0.01
delta_W_structured = B_true @ A_true  # rank-4 by construction

U2, S2, Vt2 = torch.linalg.svd(delta_W_structured, full_matrices=False)

total_energy2 = (S2 ** 2).sum()
for r in [1, 2, 4, 8, 16]:
    captured = (S2[:r] ** 2).sum() / total_energy2 * 100
    print(f"r={r:3d}: {captured:.2f}% of energy captured")

print(f"\nWith true rank={true_rank}, r={true_rank} captures 100% — perfect reconstruction!")

## The Scaling Factor ($\alpha / r$)

The scaling factor decouples the learning rate from the rank. Let's see its effect:

In [ ]:
import torch

d, k, r = 128, 128, 8
x = torch.randn(1, k)
B = torch.randn(d, r) * 0.01
A = torch.randn(r, k) * 0.01

for alpha in [4, 8, 16, 32]:
    scaling = alpha / r
    lora_out = (x @ A.T @ B.T) * scaling
    print(f"alpha={alpha:2d}, scaling={scaling:.1f}, "
          f"output norm={lora_out.norm():.4f}")

## Layer-Level Parameter Counts

How many parameters does LoRA add across an entire Llama-scale model?

In [ ]:
# Llama 3.1 8B has 32 transformer layers
num_layers = 32
d_model = 4096
d_ffn = 14336  # MLP intermediate size
r = 16

# Attention-only LoRA
attn_params = num_layers * 4 * (2 * d_model * r)
# Attention + MLP LoRA
mlp_params = num_layers * 3 * ((d_model * r) + (d_ffn * r))
total_attn = attn_params
total_all = attn_params + mlp_params

print(f"Attention-only LoRA: {total_attn:>12,} params")
print(f"Attention + MLP LoRA: {total_all:>11,} params")
print(f"Full model:           8,000,000,000 params")
print(f"Attn LoRA %:          {total_attn/8e9*100:.2f}%")
print(f"All LoRA %:           {total_all/8e9*100:.2f}%")

## Complete Training Example

Let's fine-tune a small model (SmolLM2-135M) with LoRA and inspect the matrices before and after training.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch

model_name = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float32
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### Inspect LoRA matrices before training

$B$ should be all zeros, making $\Delta W = BA = 0$.

In [ ]:
# Grab the LoRA matrices from the first layer's q_proj
q_lora = model.model.model.layers[0].self_attn.q_proj
lora_A = q_lora.lora_A.default.weight.data
lora_B = q_lora.lora_B.default.weight.data

print(f"lora_A shape: {lora_A.shape}, norm: {lora_A.norm():.4f}")
print(f"lora_B shape: {lora_B.shape}, norm: {lora_B.norm():.4f}")
print(f"ΔW = B·A norm: {(lora_B @ lora_A).norm():.4f}")

### Train on a small dataset

In [ ]:
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

# Simple dataset for demonstration
data = Dataset.from_list([
    {"text": "The capital of France is Paris."},
    {"text": "Machine learning models learn from data."},
    {"text": "PyTorch is a deep learning framework."},
    {"text": "Transformers use self-attention mechanisms."},
] * 50)  # repeat for a few epochs worth of data

tokenizer.pad_token = tokenizer.eos_token
training_args = SFTConfig(
    output_dir="./lora-demo",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=1e-3,
    logging_steps=25,
    max_length=64,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=data,
    processing_class=tokenizer,
)

trainer.train()

### Inspect LoRA matrices after training

$B$ should no longer be zero — the model has learned a low-rank update.

In [ ]:
lora_A_after = q_lora.lora_A.default.weight.data
lora_B_after = q_lora.lora_B.default.weight.data
delta_W = lora_B_after @ lora_A_after

print(f"lora_A norm: {lora_A_after.norm():.4f}")
print(f"lora_B norm: {lora_B_after.norm():.4f}")
print(f"ΔW = B·A norm: {delta_W.norm():.4f}")
print(f"ΔW shape: {delta_W.shape}")
print(f"ΔW rank: ≤ {lora_config.r} (by construction)")

## Merging LoRA into the Base Model

Merging is just matrix addition: $W_{\text{merged}} = W_0 + \frac{\alpha}{r} \cdot BA$

After merging, the model is a regular model — no adapter overhead at inference.

In [ ]:
# Save the original weight and LoRA matrices before merging
W0 = q_lora.weight.data.clone()
scaling = lora_config.lora_alpha / lora_config.r
delta_W_scaled = (lora_B_after @ lora_A_after) * scaling

# Manually compute merged weight
W_manual = W0 + delta_W_scaled

# Use PEFT's merge_and_unload
merged_model = model.merge_and_unload()

# Compare
W_peft = merged_model.model.layers[0].self_attn.q_proj.weight.data

print(f"Manual merge matches PEFT: {torch.allclose(W_manual, W_peft, atol=1e-5)}")
print(f"\nW0 norm:       {W0.norm():.4f}")
print(f"ΔW scaled norm: {delta_W_scaled.norm():.4f}")
print(f"W_merged norm:  {W_peft.norm():.4f}")

## Merged vs. Unmerged Inference Benchmark

When you don't merge, every forward pass computes two paths. Let's measure the overhead.

In [ ]:
import torch
import time

d, k, r = 4096, 4096, 16
x = torch.randn(32, k)  # batch of 32

W0 = torch.randn(d, k)
B = torch.randn(d, r)
A = torch.randn(r, k)
W_merged = W0 + B @ A

# Benchmark: merged (single matmul) vs unmerged (two paths)
def bench_merged(x, W, n=1000):
    for _ in range(n):
        _ = x @ W.T

def bench_unmerged(x, W0, A, B, n=1000):
    for _ in range(n):
        _ = x @ W0.T + x @ A.T @ B.T

t0 = time.perf_counter()
bench_merged(x, W_merged)
t_merged = time.perf_counter() - t0

t0 = time.perf_counter()
bench_unmerged(x, W0, A, B)
t_unmerged = time.perf_counter() - t0

print(f"Merged:   {t_merged:.3f}s")
print(f"Unmerged: {t_unmerged:.3f}s")
print(f"Overhead: {(t_unmerged/t_merged - 1)*100:.1f}%")

## The Merge in Detail

Tracing exactly what `merge_and_unload` does — it's just matrix addition with a scaling factor.

In [ ]:
import torch

# Simulating the merge process
d, k, r = 512, 512, 8
alpha, rank = 16, 8
scaling = alpha / rank

# Pretrained weight (frozen during training)
W0 = torch.randn(d, k)

# Learned LoRA matrices
B = torch.randn(d, r) * 0.01
A = torch.randn(r, k) * 0.01

# Step 1: compute the low-rank update
delta_W = B @ A  # (d × r) @ (r × k) = (d × k)
print(f"ΔW shape: {delta_W.shape}")

# Step 2: apply the scaling factor
delta_W_scaled = delta_W * scaling
print(f"Scaling factor (α/r): {scaling}")

# Step 3: add to original weights
W_merged = W0 + delta_W_scaled
print(f"W0 norm:       {W0.norm():.2f}")
print(f"ΔW norm:       {delta_W_scaled.norm():.2f}")
print(f"W_merged norm: {W_merged.norm():.2f}")

## Summary

LoRA in a nutshell:
1. **Freeze** the pretrained weights $W_0$
2. **Learn** two small matrices $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$ where $r \ll \min(d, k)$
3. **Forward pass**: $h = W_0 x + \frac{\alpha}{r} \cdot BAx$
4. **Merge** (optional): $W_{\text{merged}} = W_0 + \frac{\alpha}{r} \cdot BA$ for zero overhead at inference

That's the whole algorithm. The rest is engineering.